# Spam Email Detection System
**Machine Learning Internship Project - SoftGrowTech**  
Simple & High Accuracy (98%+) Model using TF-IDF + Naive Bayes

In [2]:
# ============================================================
# Cell 1: Install Required Libraries & Import Dependencies
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import joblib
import warnings
warnings.filterwarnings('ignore')

# NLTK
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

In [5]:
df = pd.read_csv('emails.csv')   # adjust path if needed
print(f"\n Shape          : {df.shape}")
print(f"   Total emails   : {len(df):,}")
print(f"   Feature columns: {len(df.columns) - 2:,}  (word frequencies)")
print(f"\n First 5 rows (first 8 columns shown):")
display(df.iloc[:5, :8])

print(f"\n📋 Label Distribution  (Prediction column):")
vc = df['Prediction'].value_counts()
for label, count in vc.items():
    name = "Ham (Legitimate)" if label == 0 else "Spam"
    print(f"   {label} → {name:20s} : {count:,}  ({count/len(df)*100:.1f}%)")

print(f"\n Missing values : {df.isnull().sum().sum()}")


 Shape          : (5172, 3002)
   Total emails   : 5,172
   Feature columns: 3,000  (word frequencies)

 First 5 rows (first 8 columns shown):


,Email No.,the,to,ect,and,for,of,a
0,Email 1,0,0,1,0,0,0,2
1,Email 2,8,13,24,6,6,2,102
2,Email 3,0,0,1,0,0,0,8
3,Email 4,0,5,22,0,5,1,51
4,Email 5,7,6,17,1,5,2,57



📋 Label Distribution  (Prediction column):
   0 → Ham (Legitimate)     : 3,672  (71.0%)
   1 → Spam                 : 1,500  (29.0%)

 Missing values : 0


In [13]:
# ============================================================
# Cell 3: Exploratory Data Analysis (EDA)
# ============================================================

feature_cols = [c for c in df.columns if c not in ['Email No.', 'Prediction']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(" Spam Email Detection — EDA", fontsize=16,
             fontweight='bold', y=1.02)

palette = {0: '#2ecc71', 1: '#e74c3c'}
label_names = {0: 'Ham', 1: 'Spam'}

# ── Plot 1: Class Distribution ──────────────────────────────
counts = df['Prediction'].value_counts().sort_index()
bars   = axes[0].bar([label_names[i] for i in counts.index],
                     counts.values,
                     color=[palette[i] for i in counts.index],
                     edgecolor='black', linewidth=0.8)
axes[0].set_title("Class Distribution", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20, str(val),
                 ha='center', fontsize=12, fontweight='bold')

# ── Plot 2: Total Word Count per Email ──────────────────────
df['word_count'] = df[feature_cols].sum(axis=1)
for label in [0, 1]:
    subset = df[df['Prediction'] == label]['word_count']
    axes[1].hist(subset, bins=50, alpha=0.65,
                 label=label_names[label],
                 color=palette[label], edgecolor='white')
axes[1].set_title("Total Word Count per Email", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Total Word Count")
axes[1].set_ylabel("Frequency")
axes[1].legend()
axes[1].set_xlim(0, df['word_count'].quantile(0.99))  # remove extreme outliers

# ── Plot 3: Unique Words per Email ──────────────────────────
df['unique_words'] = (df[feature_cols] > 0).sum(axis=1)
data_box = [df[df['Prediction']==0]['unique_words'],
            df[df['Prediction']==1]['unique_words']]
bp = axes[2].boxplot(data_box, patch_artist=True, notch=True,
                     labels=['Ham', 'Spam'])
for patch, color in zip(bp['boxes'], ['#2ecc71', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title("Unique Words per Email", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Label")
axes[2].set_ylabel("Unique Word Count")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Summary Statistics ───────────────────────────────────────
print("\n Word Count Statistics by Class:")
print(df.groupby('Prediction')[['word_count','unique_words']].describe().round(2))